In [1]:
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DEM_Milestone3") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark Started")

Spark Started


In [8]:
import zipfile
import os

zip_path = "/content/ingestion_layer.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/extracted")

print("ZIP Extracted Successfully")

ZIP Extracted Successfully


In [9]:
import os

parquet_files = []

for root, dirs, files in os.walk("/content/extracted"):
    for file in files:
        if file.endswith(".parquet"):
            parquet_files.append(os.path.join(root, file))

print("Number of parquet files:", len(parquet_files))
print("\nFirst few files:")

for p in parquet_files[:5]:
    print(p)

Number of parquet files: 192

First few files:
/content/extracted/ingestion_layer/partitioned_data/year=2024/month=4/part-00002-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet
/content/extracted/ingestion_layer/partitioned_data/year=2024/month=4/part-00005-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet
/content/extracted/ingestion_layer/partitioned_data/year=2024/month=4/part-00001-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet
/content/extracted/ingestion_layer/partitioned_data/year=2024/month=4/part-00004-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet
/content/extracted/ingestion_layer/partitioned_data/year=2024/month=4/part-00003-a89b91bc-951d-4be0-a71a-876c2ab03c30.c000.snappy.parquet


In [10]:
df = spark.read.parquet(*parquet_files)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [11]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 8844800
Columns: 20


In [12]:
df.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- datetime: date (nullable = true)
 |-- at_c: double (nullable = true)
 |-- rh_percent: double (nullable = true)
 |-- ws_m_s: double (nullable = true)
 |-- wd_deg: double (nullable = true)
 |-- rf_mm: double (nullable = true)
 |-- tot_rf_mm: double (nullable = true)
 |-- sr_w_mt2: double (nullable = true)
 |-- bp_mmhg: double (nullable = true)
 |-- vws_m_s: double (nullable = true)
 |-- pollutant: string (nullable = true)
 |-- value: double (nullable = true)
 |-- station: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)



In [13]:
df.show(5, truncate=False)

+----------+-----+-----+-----------------------------+-------------------------------+----------+----+----------+------+------+-----+---------+--------+-------+-------+---------+------+-----------------------------+---+----+
|station_id|state|city |station_name                 |timestamp                      |datetime  |at_c|rh_percent|ws_m_s|wd_deg|rf_mm|tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value |station                      |day|hour|
+----------+-----+-----+-----------------------------+-------------------------------+----------+----+----------+------+------+-----+---------+--------+-------+-------+---------+------+-----------------------------+---+----+
|site_1427 |Delhi|Delhi|Najafgarh, Delhi - DPCC      |2024-03-01T00:00:00.000000+0000|2024-03-01|14.0|78.0      |0.4   |210.0 |0.0  |0.0      |3.0     |994.0  |NULL   |pm10     |178.0 |Najafgarh, Delhi - DPCC      |1  |0   |
|site_105  |Delhi|Delhi|North Campus, DU, Delhi - IMD|2024-03-01T00:15:00.000000+0000|2024-03-01|NUL

In [14]:
from pyspark.sql.functions import col, when, count

missing = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing.show(truncate=False)

+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+---+----+
|station_id|state|city|station_name|timestamp|datetime|at_c   |rh_percent|ws_m_s |wd_deg |rf_mm  |tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value|station|day|hour|
+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+---+----+
|0         |0    |0   |0           |0        |0       |2532923|1519086   |1531562|1836378|4006105|801929   |1610725 |3022601|6546511|0        |0    |0      |0  |0   |
+----------+-----+----+------------+---------+--------+-------+----------+-------+-------+-------+---------+--------+-------+-------+---------+-----+-------+---+----+



In [15]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

Total Rows: 8844800
Unique Rows: 8844800
Duplicates: 0


In [16]:
df.describe().show(truncate=False)

+-------+----------+-------+-------+--------------------------------------+-------------------------------+-----------------+------------------+------------------+-----------------+--------------------+-------------------+------------------+-----------------+--------------------+---------+-----------------+--------------------------------------+------------------+------------------+
|summary|station_id|state  |city   |station_name                          |timestamp                      |at_c             |rh_percent        |ws_m_s            |wd_deg           |rf_mm               |tot_rf_mm          |sr_w_mt2          |bp_mmhg          |vws_m_s             |pollutant|value            |station                               |day               |hour              |
+-------+----------+-------+-------+--------------------------------------+-------------------------------+-----------------+------------------+------------------+-----------------+--------------------+-------------------+------

In [17]:
from pyspark.sql.functions import col

numeric_cols = [
    "at_c",
    "rh_percent",
    "ws_m_s",
    "wd_deg",
    "tot_rf_mm",
    "sr_w_mt2",
    "bp_mmhg"
]

medians = {}

for c in numeric_cols:
    medians[c] = df.approxQuantile(c, [0.5], 0.01)[0]

print(medians)

{'at_c': 27.1, 'rh_percent': 65.0, 'ws_m_s': 0.6, 'wd_deg': 189.0, 'tot_rf_mm': 0.0, 'sr_w_mt2': 17.0, 'bp_mmhg': 983.0}


In [18]:
clean_df = df

for c in numeric_cols:
    clean_df = clean_df.fillna(
        medians[c],
        subset=[c]
    )

clean_df = clean_df.fillna(
    0,
    subset=["rf_mm"]
)

In [19]:
from pyspark.sql.functions import count, when

missing_after = clean_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in clean_df.columns
])

missing_after.show(truncate=False)

+----------+-----+----+------------+---------+--------+----+----------+------+------+-----+---------+--------+-------+-------+---------+-----+-------+---+----+
|station_id|state|city|station_name|timestamp|datetime|at_c|rh_percent|ws_m_s|wd_deg|rf_mm|tot_rf_mm|sr_w_mt2|bp_mmhg|vws_m_s|pollutant|value|station|day|hour|
+----------+-----+----+------------+---------+--------+----+----------+------+------+-----+---------+--------+-------+-------+---------+-----+-------+---+----+
|0         |0    |0   |0           |0        |0       |0   |0         |0     |0     |0    |0        |0       |0      |6546511|0        |0    |0      |0  |0   |
+----------+-----+----+------------+---------+--------+----+----------+------+------+-----+---------+--------+-------+-------+---------+-----+-------+---+----+



In [20]:
from pyspark.sql.functions import when

clean_df = clean_df.withColumn(
    "aqi_category",
    when(col("value") <= 50, "Good")
    .when(col("value") <= 100, "Moderate")
    .when(col("value") <= 200, "Poor")
    .otherwise("Severe")
)

In [21]:
from pyspark.sql.functions import month

clean_df = clean_df.withColumn(
    "month",
    month(col("datetime"))
)

In [22]:
from pyspark.sql.functions import when

clean_df = clean_df.withColumn(
    "season",
    when(col("month").isin([12,1,2]), "Winter")
    .when(col("month").isin([3,4,5]), "Summer")
    .when(col("month").isin([6,7,8,9]), "Monsoon")
    .otherwise("Post-Monsoon")
)

In [24]:
from pyspark.sql.functions import year, month

clean_df = clean_df.withColumn(
    "year",
    year("datetime")
)

clean_df = clean_df.withColumn(
    "month",
    month("datetime")
)

clean_df.select(
    "datetime",
    "year",
    "month"
).show(5)

+----------+----+-----+
|  datetime|year|month|
+----------+----+-----+
|2024-03-01|2024|    3|
|2024-03-01|2024|    3|
|2024-03-01|2024|    3|
|2024-03-01|2024|    3|
|2024-03-01|2024|    3|
+----------+----+-----+
only showing top 5 rows


In [25]:
clean_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/content/cleaned_data")

In [26]:
print(clean_df.columns)

['station_id', 'state', 'city', 'station_name', 'timestamp', 'datetime', 'at_c', 'rh_percent', 'ws_m_s', 'wd_deg', 'rf_mm', 'tot_rf_mm', 'sr_w_mt2', 'bp_mmhg', 'vws_m_s', 'pollutant', 'value', 'station', 'day', 'hour', 'aqi_category', 'month', 'season', 'year']
